In [14]:
# Task 4.1
import sqlite3

conn = sqlite3.connect('CLINIC.db')
cursor = conn.cursor()

with conn:
    cursor.executescript("""
        DROP TABLE IF EXISTS Patient;
        DROP TABLE IF EXISTS Queue;

        CREATE TABLE IF NOT EXISTS Patient(
            PatientID INTEGER PRIMARY KEY,
            Name TEXT NOT NULL,
            Age INT NOT NULL,
            Gender TEXT NOT NULL
        );

        CREATE TABLE IF NOT EXISTS Queue(
            ID INTEGER PRIMARY KEY AUTOINCREMENT,
            PatientID INT NOT NULL,
            Priority INT NOT NULL,
            InQueue INT NOT NULL,

            FOREIGN KEY (PatientID) REFERENCES Patient(PatientID)
        );
    """)

    conn.commit()

In [15]:
# Task 4.2
import sqlite3
import csv

conn = sqlite3.connect('CLINIC.db')
cursor = conn.cursor()

with conn:
    with open('patient.txt') as f:
        data = [i for i in csv.reader(f)]
        for item in data:
            cursor.execute('INSERT INTO Patient VALUES (?, ?, ?, ?)', item)

    with open('queue.txt') as f:
        data = [i for i in csv.reader(f)]
        for item in data:
            cursor.execute('INSERT INTO Queue(PatientID, Priority, InQueue) VALUES (?, ?, ?)', item)

    conn.commit()

In [21]:
# Task 4.3
name = input('name?: ')

import sqlite3

conn = sqlite3.connect('CLINIC.db')
cursor = conn.cursor()

with conn:
    cursor.execute('SELECT Name, Age, Gender FROM Patient WHERE Name = ?', (name,))
    result = cursor.fetchall()

    if result == []:
        print('Not found')
    else:
        print(result[0])

('John', 34, 'Male')


In [25]:
# Task 4.4
from flask import Flask, render_template
import sqlite3

app = Flask(__name__, template_folder='Task4_4_Akshat')

@app.route('/')
def index():
    conn = sqlite3.connect('CLINIC.db')
    cursor = conn.cursor()

    with conn:
        cursor.execute('SELECT Q.ID, Q.Priority, P.Name, P.Age, P.Gender FROM Patient AS P JOIN Queue AS Q ON Q.PatientID = P.PatientID WHERE Q.InQueue = 1 ORDER BY Q.Priority DESC, Q.ID ASC')
        data = cursor.fetchall()

    return render_template('index.html', data=data)

app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [30/Aug/2026 14:15:17] "GET / HTTP/1.1" 200 -
